# Part 5: Advanced GNN Models
## PNA, GraphTransformer, AttentiveFP, and Fused Variants

This notebook trains advanced GNN architectures and explores the **fused** approach
where fingerprints are combined at the graph level (not per-atom):
- **PNA**: 4 aggregators + 3 scalers + residual connections
- **GraphTransformer**: Global self-attention + edge bias
- **AttentiveFP**: Graph attention + GRU + attentive readout (following OpenDrugAI reference)
- **FusedGNN**: Lightweight graph (32-dim) + fingerprint branch

## AttentiveFP Pipeline (following reference principles)
```
XO DATASET → Data standardization → Deduplication → Active/inactive labels
→ DATA SPLIT (scaffold for AttentiveFP) → TRAIN/TEST
→ Stratified CV / tuning → GCN / AttentiveFP / Morgan → Comparison
→ Final independent test → ROC-AUC + PR-AUC + MCC + F1 + sensitivity + specificity
→ scaffold/generalization test
```

In [ ]:
# @title 1. Setup
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    DEVICE = torch.device('cpu')
    print("No GPU - using CPU")

train_df = pd.read_csv('data/train.csv')
val_df = pd.read_csv('data/val.csv')
test_df = pd.read_csv('data/test.csv')
print(f"Data: Train={len(train_df)} Val={len(val_df)} Test={len(test_df)}")

In [ ]:
# @title 2. Scaffold Split for AttentiveFP (Generalization Test)
from vegfr2.data import scaffold_split

# Load full dataset for scaffold split
full_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
print(f"Full dataset: {len(full_df)} compounds")

# Scaffold split: ensures test set has different molecular scaffolds
scaffold_train, scaffold_val, scaffold_test = scaffold_split(full_df, seed=42)

print(f"\nScaffold Split:")
print(f"  Train: {len(scaffold_train)} ({scaffold_train['active'].mean():.1%} active)")
print(f"  Val:   {len(scaffold_val)} ({scaffold_val['active'].mean():.1%} active)")
print(f"  Test:  {len(scaffold_test)} ({scaffold_test['active'].mean():.1%} active)")

# Check scaffold overlap
train_scaffolds = set(scaffold_train['smiles'].apply(
    lambda s: __import__('rdkit.Chem.Scaffolds.MurckoScaffold', fromlist=['MurckoScaffoldSmiles']).MurckoScaffoldSmiles(smiles=s)
))
test_scaffolds = set(scaffold_test['smiles'].apply(
    lambda s: __import__('rdkit.Chem.Scaffolds.MurckoScaffold', fromlist=['MurckoScaffoldSmiles']).MurckoScaffoldSmiles(smiles=s)
))
overlap = train_scaffolds & test_scaffolds
print(f"\n  Train scaffolds: {len(train_scaffolds)}")
print(f"  Test scaffolds:  {len(test_scaffolds)}")
print(f"  Overlap: {len(overlap)} ({len(overlap)/len(test_scaffolds)*100:.1f}% of test)")

In [ ]:
# @title 3. Create DataLoaders
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from vegfr2.features import mol_to_graph, mol_to_graph_with_fps

def make_enriched_loader(df, batch_size=128, shuffle=False):
    data_list = []
    for s, y in zip(df['smiles'], df['active'].astype(int)):
        try:
            g = mol_to_graph_with_fps(s, use_morgan=True, use_maccs=True)
            data = Data(x=g['node_feats'], edge_index=g['edge_index'],
                       edge_attr=g['edge_feats'],
                       y=torch.tensor([y], dtype=torch.float32))
            data_list.append(data)
        except Exception:
            pass
    return DataLoader(data_list, batch_size=batch_size, shuffle=shuffle)

def make_graph_only_loader(df, batch_size=128, shuffle=False):
    data_list = []
    for s, y in zip(df['smiles'], df['active'].astype(int)):
        try:
            g = mol_to_graph(s)
            data = Data(x=g['node_feats'], edge_index=g['edge_index'],
                       edge_attr=g['edge_feats'],
                       y=torch.tensor([y], dtype=torch.float32))
            data_list.append(data)
        except Exception:
            pass
    return DataLoader(data_list, batch_size=batch_size, shuffle=shuffle)

def make_fused_loader(df, fps, batch_size=128, shuffle=False):
    data_list = []
    for s, y, fp in zip(df['smiles'], df['active'].astype(int), fps):
        try:
            g = mol_to_graph(s)
            data = Data(x=g['node_feats'], edge_index=g['edge_index'],
                       edge_attr=g['edge_feats'],
                       y=torch.tensor([y], dtype=torch.float32),
                       fingerprint=torch.tensor(fp, dtype=torch.float32).unsqueeze(0))
            data_list.append(data)
        except Exception:
            pass
    return DataLoader(data_list, batch_size=batch_size, shuffle=shuffle)

# Standard loaders (random split)
enriched_train = make_enriched_loader(train_df, shuffle=True)
enriched_val = make_enriched_loader(val_df)
enriched_test = make_enriched_loader(test_df)

# Scaffold split loaders (for AttentiveFP generalization test)
scaffold_enriched_train = make_enriched_loader(scaffold_train, shuffle=True)
scaffold_enriched_val = make_enriched_loader(scaffold_val)
scaffold_enriched_test = make_enriched_loader(scaffold_test)

graph_train = make_graph_only_loader(train_df, shuffle=True)
graph_val = make_graph_only_loader(val_df)
graph_test = make_graph_only_loader(test_df)

print("Loaders created (standard + scaffold)")

In [ ]:
# @title 4. Forward Helper (handles all model types)
from vegfr2.gnn_pyg import graph_forward

def train_gnn(model_name, train_loader, val_loader, test_loader, in_dim=2246, epochs=100, patience=15, lr=0.001):
    torch.manual_seed(42)
    hidden = 64 if model_name == 'mpnn' else (200 if model_name == 'attentive_fp' else 128)
    model = build_pyg_model(model_name, in_dim=in_dim, hidden=hidden, layers=3, heads=8, dropout=0.3).to(DEVICE)
    n_params = sum(p.numel() for p in model.parameters())
    print(f"  {model_name}: {n_params:,} params")
    
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)
    loss_fn = nn.BCEWithLogitsLoss()
    
    best_auc, best_state, wait = -1.0, None, 0
    
    for epoch in range(1, epochs + 1):
        model.train()
        for batch in train_loader:
            batch = batch.to(DEVICE)
            logits = graph_forward(model, batch)
            loss = loss_fn(logits.squeeze(), batch.y)
            opt.zero_grad()
            loss.backward()
            opt.step()
        scheduler.step()
        
        model.eval()
        val_probs, val_true = [], []
        with torch.no_grad():
            for batch in val_loader:
                batch = batch.to(DEVICE)
                logits = graph_forward(model, batch)
                val_probs.extend(torch.sigmoid(logits).squeeze().cpu().numpy())
                val_true.extend(batch.y.squeeze().cpu().numpy().astype(int))
        
        val_auc = classification_metrics(val_true, val_probs).get('auc') or 0.0
        if val_auc > best_auc:
            best_auc = val_auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1
            if wait >= patience:
                break
    
    if best_state:
        model.load_state_dict(best_state)
    model.to(DEVICE).eval()
    
    test_probs, test_true = [], []
    with torch.no_grad():
        for batch in test_loader:
            batch = batch.to(DEVICE)
            logits = graph_forward(model, batch)
            test_probs.extend(torch.sigmoid(logits).squeeze().cpu().numpy())
            test_true.extend(batch.y.squeeze().cpu().numpy().astype(int))
    
    return classification_metrics(test_true, test_probs), model

In [ ]:
# @title 5. Train PNA and GraphTransformer (Enriched, Random Split)
from vegfr2.gnn_pyg import build_pyg_model
from vegfr2.metrics import classification_metrics

results = {}

for name in ['pna', 'graph_transformer']:
    print(f"\nTraining {name.upper()} (enriched, random split)...")
    metrics, model = train_gnn(name, enriched_train, enriched_val, enriched_test)
    results[f'{name}_enriched'] = metrics
    print(f"  AUC={metrics.get('auc', 0):.4f} ACC={metrics['acc']:.4f} MCC={metrics['mcc']:.4f}")

In [ ]:
# @title 6. Train AttentiveFP (Following Reference: Scaffold Split)
# AttentiveFP uses scaffold split for generalization testing
# Reference hyperparameters from OpenDrugAI:
#   hidden=200, lr=1e-3, weight_decay=1e-4, dropout=0.2, T=2, radius=2

print("="*60)
print("AttentiveFP - Scaffold Split (Generalization Test)")
print("Following OpenDrugAI/AttentiveFP reference")
print("="*60)

# Train with scaffold split
afp_scaffold_metrics, afp_scaffold_model = train_gnn(
    'attentive_fp',
    scaffold_enriched_train,
    scaffold_enriched_val,
    scaffold_enriched_test,
    in_dim=2246,
    epochs=100,
    patience=15,
    lr=0.001,
)
results['attentive_fp_scaffold'] = afp_scaffold_metrics
print(f"\n  Scaffold split AUC: {afp_scaffold_metrics.get('auc', 0):.4f}")
print(f"  Scaffold split MCC: {afp_scaffold_metrics['mcc']:.4f}")

# Also train with random split for comparison
print("\n" + "="*60)
print("AttentiveFP - Random Split (Baseline)")
print("="*60)

afp_random_metrics, afp_random_model = train_gnn(
    'attentive_fp',
    enriched_train,
    enriched_val,
    enriched_test,
    in_dim=2246,
    epochs=100,
    patience=15,
    lr=0.001,
)
results['attentive_fp_random'] = afp_random_metrics
print(f"\n  Random split AUC: {afp_random_metrics.get('auc', 0):.4f}")
print(f"  Random split MCC: {afp_random_metrics['mcc']:.4f}")

In [ ]:
# @title 7. AttentiveFP Detailed Metrics
from sklearn.metrics import roc_auc_score, precision_recall_curve, auc, f1_score

def compute_detailed_metrics(y_true, y_prob, threshold=0.5):
    y_pred = (np.array(y_prob) >= threshold).astype(int)
    y_true = np.array(y_true)
    
    tp = ((y_pred == 1) & (y_true == 1)).sum()
    tn = ((y_pred == 0) & (y_true == 0)).sum()
    fp = ((y_pred == 1) & (y_true == 0)).sum()
    fn = ((y_pred == 0) & (y_true == 1)).sum()
    
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    pr_auc = auc(recall, precision)
    
    return {
        'ROC-AUC': roc_auc_score(y_true, y_prob) if len(set(y_true)) >= 2 else 0,
        'PR-AUC': pr_auc,
        'MCC': classification_metrics(y_true.tolist(), y_prob)['mcc'],
        'F1': f1_score(y_true, y_pred),
        'Sensitivity': sens,
        'Specificity': spec,
    }

# Get test predictions for AttentiveFP (scaffold)
afp_scaffold_model.eval()
afp_scaffold_probs = []
afp_scaffold_true = []
with torch.no_grad():
    for batch in scaffold_enriched_test:
        batch = batch.to(DEVICE)
        logits = graph_forward(afp_scaffold_model, batch)
        afp_scaffold_probs.extend(torch.sigmoid(logits).squeeze().cpu().numpy())
        afp_scaffold_true.extend(batch.y.squeeze().cpu().numpy().astype(int))

scaffold_detailed = compute_detailed_metrics(afp_scaffold_true, afp_scaffold_probs)

print("\n" + "="*60)
print("AttentiveFP (Scaffold Split) - Detailed Metrics")
print("="*60)
for metric, value in scaffold_detailed.items():
    print(f"  {metric:<15} {value:.4f}")

# Also get random split metrics
afp_random_model.eval()
afp_random_probs = []
afp_random_true = []
with torch.no_grad():
    for batch in enriched_test:
        batch = batch.to(DEVICE)
        logits = graph_forward(afp_random_model, batch)
        afp_random_probs.extend(torch.sigmoid(logits).squeeze().cpu().numpy())
        afp_random_true.extend(batch.y.squeeze().cpu().numpy().astype(int))

random_detailed = compute_detailed_metrics(afp_random_true, afp_random_probs)

print("\n" + "="*60)
print("AttentiveFP (Random Split) - Detailed Metrics")
print("="*60)
for metric, value in random_detailed.items():
    print(f"  {metric:<15} {value:.4f}")

# Comparison
print("\n" + "="*60)
print("Scaffold vs Random Split Comparison")
print("="*60)
print(f"{'Metric':<15} {'Random':>10} {'Scaffold':>10} {'Gap':>10}")
print("-"*45)
for metric in scaffold_detailed:
    r = random_detailed[metric]
    s = scaffold_detailed[metric]
    gap = r - s
    print(f"{metric:<15} {r:>10.4f} {s:>10.4f} {gap:>+10.4f}")
print("\nLarger gap = worse generalization (model overfits to training scaffolds)")

In [ ]:
# @title 8. Fused Approach: Graph + Fingerprint at Graph Level
from vegfr2.features import smiles_to_morgan, smiles_to_maccs
from vegfr2.models.fused_gnn import FusedGIN, FusedGAT

# Extract fingerprints for fusion
X_train_fp = np.hstack([
    np.vstack([smiles_to_morgan(s) for s in train_df['smiles']]),
    np.vstack([smiles_to_maccs(s) for s in train_df['smiles']])
]).astype(np.float32)

X_val_fp = np.hstack([
    np.vstack([smiles_to_morgan(s) for s in val_df['smiles']]),
    np.vstack([smiles_to_maccs(s) for s in val_df['smiles']])
]).astype(np.float32)

X_test_fp = np.hstack([
    np.vstack([smiles_to_morgan(s) for s in test_df['smiles']]),
    np.vstack([smiles_to_maccs(s) for s in test_df['smiles']])
]).astype(np.float32)

y_train = train_df['active'].values.astype(int)
y_val = val_df['active'].values.astype(int)
y_test = test_df['active'].values.astype(int)

fused_train = make_fused_loader(train_df, X_train_fp, shuffle=True)
fused_val = make_fused_loader(val_df, X_val_fp)
fused_test = make_fused_loader(test_df, X_test_fp)

# Train FusedGIN
print("\nTraining FusedGIN (graph 32-dim + FP 2214-dim)...")
fused_model = FusedGIN(in_dim=32, hidden=128, layers=3, fp_dim=2214, out_dim=1, dropout=0.3).to(DEVICE)
n_params = sum(p.numel() for p in fused_model.parameters())
print(f"  FusedGIN: {n_params:,} params")

opt = torch.optim.AdamW(fused_model.parameters(), lr=0.001, weight_decay=1e-4)
loss_fn = nn.BCEWithLogitsLoss()

best_auc, best_state, wait = -1.0, None, 0
for epoch in range(1, 101):
    fused_model.train()
    for batch in fused_train:
        batch = batch.to(DEVICE)
        logits = fused_model(batch.x, batch.edge_index, batch.batch, batch.fingerprint)
        loss = loss_fn(logits.squeeze(), batch.y)
        opt.zero_grad()
        loss.backward()
        opt.step()
    
    fused_model.eval()
    val_probs, val_true = [], []
    with torch.no_grad():
        for batch in fused_val:
            batch = batch.to(DEVICE)
            logits = fused_model(batch.x, batch.edge_index, batch.batch, batch.fingerprint)
            val_probs.extend(torch.sigmoid(logits).squeeze().cpu().numpy())
            val_true.extend(batch.y.squeeze().cpu().numpy().astype(int))
    
    val_auc = classification_metrics(val_true, val_probs).get('auc') or 0.0
    if val_auc > best_auc:
        best_auc = val_auc
        best_state = {k: v.cpu().clone() for k, v in fused_model.state_dict().items()}
        wait = 0
    else:
        wait += 1
        if wait >= 15:
            break

if best_state:
    fused_model.load_state_dict(best_state)
fused_model.to(DEVICE).eval()

test_probs, test_true = [], []
with torch.no_grad():
    for batch in fused_test:
        batch = batch.to(DEVICE)
        logits = fused_model(batch.x, batch.edge_index, batch.batch, batch.fingerprint)
        test_probs.extend(torch.sigmoid(logits).squeeze().cpu().numpy())
        test_true.extend(batch.y.squeeze().cpu().numpy().astype(int))

results['fused_gin'] = classification_metrics(test_true, test_probs)
print(f"  FusedGIN AUC: {results['fused_gin'].get('auc', 0):.4f}")

In [ ]:
# @title 9. Final Comparison
print("\n" + "="*70)
print("ADVANCED GNN COMPARISON")
print("="*70)

header = f"{'Model':<35} {'ACC':>6} {'SEN':>6} {'SPE':>6} {'MCC':>6} {'AUC':>6}"
print(header)
print("-"*70)

for name, m in sorted(results.items(), key=lambda x: x[1].get('auc') or 0, reverse=True):
    auc_str = f"{m['auc']:.4f}" if m.get('auc') is not None else "N/A"
    print(f"{name:<35} {m['acc']:.4f} {m['sen']:.4f} {m['spe']:.4f} {m['mcc']:.4f} {auc_str:>6}")

print("\nKey observations:")
print("  - AttentiveFP (scaffold) should show lower AUC than (random)")
print("  - This gap indicates how well the model generalizes to new scaffolds")
print("  - If gap is large, the model is overfitting to training scaffolds")

In [ ]:
# @title 10. Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: All models
names = list(results.keys())
aucs = [results[n].get('auc', 0) for n in names]
colors = plt.cm.Set2(np.linspace(0, 1, len(names)))

bars = axes[0].barh(names, aucs, color=colors)
axes[0].set_xlabel('AUC')
axes[0].set_title('Advanced GNN Models')
axes[0].set_xlim(0.5, 1.0)

for bar, auc in zip(bars, aucs):
    axes[0].text(auc + 0.01, bar.get_y() + bar.get_height()/2, f'{auc:.4f}', va='center')

# Plot 2: AttentiveFP scaffold vs random
metrics_to_plot = ['ROC-AUC', 'PR-AUC', 'MCC', 'F1', 'Sensitivity', 'Specificity']
random_vals = [random_detailed[m] for m in metrics_to_plot]
scaffold_vals = [scaffold_detailed[m] for m in metrics_to_plot]

x = np.arange(len(metrics_to_plot))
width = 0.35

axes[1].bar(x - width/2, random_vals, width, label='Random Split', color='#4CAF50')
axes[1].bar(x + width/2, scaffold_vals, width, label='Scaffold Split', color='#FF9800')
axes[1].set_ylabel('Score')
axes[1].set_title('AttentiveFP: Random vs Scaffold Split')
axes[1].set_xticks(x)
axes[1].set_xticklabels(metrics_to_plot, rotation=45, ha='right')
axes[1].legend()
axes[1].set_ylim(0, 1)

plt.tight_layout()
plt.savefig('images/advanced_gnn_comparison.png', dpi=150, bbox_inches='tight')
plt.show()